# Clase 062 — Clasificación binaria con dígitos

Primer clasificador binario "de verdad": ¿este dígito es un **5** o no? Entrenamos un `SGDClassifier`, validamos con `StratifiedKFold` y descubrimos por qué la **accuracy sola miente** cuando las clases están desbalanceadas.

> Nota: usamos `load_digits()` (imágenes 8×8, sin conexión a internet) en lugar de MNIST 28×28, pero el planteo es idéntico.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.base import BaseEstimator

np.random.seed(42)

digits = load_digits()
X, y = digits.data, digits.target
print('X', X.shape, 'y', y.shape)
print('clases:', np.unique(y))
print('proporcion de 5s: {:.3f}'.format((y == 5).mean()))

## 1. Cargar y explorar

Cada muestra es un vector de 64 features (8×8 píxeles). Visualizamos un dígito real para confirmar que el dataset se cargó bien.

In [ ]:
idx5 = np.where(y == 5)[0][0]
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(X[idx5].reshape(8, 8), cmap='binary')
ax.set_title(f'digito real = {y[idx5]}')
ax.axis('off')
plt.show()

assert y[idx5] == 5
print('la imagen mostrada es un 5:', bool(y[idx5] == 5))

## 2. Target binario 5 vs no-5

Construimos `y_train_5 = (y_train == 5)`. El split estratificado conserva la proporción de 5s en train y test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
y_train_5 = (y_train == 5)
y_test_5 = (y_test == 5)
print('positivos (5) en train:', int(y_train_5.sum()), '/', len(y_train_5))
print('proporcion train:', round(y_train_5.mean(), 3), '| test:', round(y_test_5.mean(), 3))

## 3. `predict` vs `decision_function`

`predict` devuelve la clase (`True`/`False`); `decision_function` devuelve el **score crudo** (distancia firmada al hiperplano). Por defecto `predict` equivale a `score > 0`.

In [ ]:
sgd = SGDClassifier(random_state=42)
sgd.fit(X_train, y_train_5)

muestra = X_test[0].reshape(1, -1)
pred = sgd.predict(muestra)[0]
score = sgd.decision_function(muestra)[0]
print('predict:', bool(pred), '| decision_function:', round(float(score), 3))

assert bool(pred) == bool(score > 0)
print('predict == (score > 0):', bool(pred) == bool(score > 0))

## 4. Validación con `cross_val_score` + `StratifiedKFold`

`StratifiedKFold` mantiene la proporción de clases en cada fold (clave con desbalanceo). El SGD debe superar holgadamente el 90%.

In [ ]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
acc_sgd = cross_val_score(sgd, X_train, y_train_5, cv=skf, scoring='accuracy', n_jobs=1)
print('accuracy 3-fold SGD:', acc_sgd.round(4))
print('promedio:', round(acc_sgd.mean(), 4))

assert acc_sgd.mean() > 0.9

## 5. El accuracy paradox: baseline trivial `Never5`

Un clasificador que **nunca** predice 5 acierta el ~90% igual, solo porque el 90% de los dígitos no son 5. Comparar contra este piso demuestra por qué la accuracy engaña.

In [ ]:
class Never5Classifier(BaseEstimator):
    def fit(self, X, y=None):
        return self
    def predict(self, X):
        return np.zeros(len(X), dtype=bool)

never5 = Never5Classifier()
acc_never = cross_val_score(never5, X_train, y_train_5, cv=skf, scoring='accuracy', n_jobs=1)
print('accuracy 3-fold Never5:', acc_never.round(4))
print('promedio:', round(acc_never.mean(), 4))

# el dummy ronda 0.9 sin haber aprendido nada
assert acc_never.mean() > 0.88
print('\nel SGD supera al baseline trivial:', bool(acc_sgd.mean() > acc_never.mean()))

## 6. Comparativa visual

Ambos rondan un accuracy alto, pero uno aprendió y el otro no. La accuracy sola no los distingue: por eso necesitamos precision/recall/F1 (clase 063).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
vals = [acc_sgd.mean(), acc_never.mean()]
ax.bar(['SGD (aprende)', 'Never5 (dummy)'], vals, color=['#37a', '#888'])
ax.set_ylabel('accuracy (3-fold)')
ax.set_ylim(0, 1.05)
ax.set_title('Accuracy paradox: el dummy tambien saca ~90%')
for i, v in enumerate(vals):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.show()

## Ejercicios

1. **decision_function.** Sobre `X_test[0]`, compará el signo de `decision_function` con el resultado de `predict` en 10 muestras distintas. ¿Siempre coincide con `score > 0`?
2. **Escalado.** Entrená el SGD con y sin `StandardScaler` (o dividiendo por 16). ¿Mejora la accuracy o la convergencia?
3. **Otro dígito.** Repetí todo para "es un 3 vs no". ¿El baseline `Never3` saca lo mismo?
4. **Más folds.** Cambiá `cv=3` por `cv=10`. ¿Baja la varianza de la estimación?

## Conclusiones

- La **accuracy** en problemas desbalanceados es traicionera: el dummy `Never5` la iguala casi sin esfuerzo.
- `decision_function` da el **score continuo** que luego permite mover el umbral (clases 065/066).
- `StratifiedKFold` conserva la proporción de clases en cada fold, imprescindible con desbalanceo.
- Comparar siempre contra un **baseline trivial**: si no le ganás, tu modelo no aprendió.
- El próximo paso son precision, recall y F1 (clase 063), que sí distinguen al SGD del dummy.